<h1 style="color:black; font-family:Arial, sans-serif;">🏡 Scraping Immobilier – Avito.ma</h1>

<hr style="border:1px solid #ccc;">

<h2 style="color:#117A65;">Data Transformation</h2>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
import mysql.connector

In [ ]:
import warnings

warnings.filterwarnings('ignore')


In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM Avito_local"
df = pd.read_sql(query, conn)
df.head()

In [ ]:
df.shape[0]

In [ ]:
print("The different Features are : ",df.columns)

In [ ]:
# Replace 'Nan' strings with actual NaN values
df = df.replace('Nan', np.nan)

In [ ]:
df = df[~df['Surface_Totale'].isna()]
print(df.shape[0])

In [ ]:
df['Ville'].unique()

In [ ]:
counts = df["Ville"].value_counts()

villes_a_garder = counts[counts >= 15].index
print("✅ Villes gardées :", list(villes_a_garder))

df = df[df["Ville"].isin(villes_a_garder)].copy()


In [ ]:
df['Ville'].value_counts()

In [ ]:
# Crée un mapping pour fusionner
mapping_villes = {
    "Mohammedia": "Mohammedia-Bouznika-Bouskoura-Dar Bouazza",
    "Bouznika": "Mohammedia-Bouznika-Bouskoura-Dar Bouazza",
    "Bouskoura": "Mohammedia-Bouznika-Bouskoura-Dar Bouazza",
    "Tit Mellil": "Autres",
    "Dar Bouazza": "Mohammedia-Bouznika-Bouskoura-Dar Bouazza",
    "Tamesna": "Autres",
    "Ain Attig": "Autres",
    "Sidi Allal El Bahraoui": "Autres",
    "Mehdia": "Kénitra-Mehdia",
    "Kénitra": "Kénitra-Mehdia",
    "Larache": "Tanger-Tétouan",
    "Martil": "Tanger-Tétouan",
    "Tétouan": "Tanger-Tétouan",
    "Tanger": "Tanger-Tétouan",
    "Oujda": "Autres",
    "Safi": "Autres",
    "Béni Mellal": "Autres",
    "Taza": "Autres",
    "Laâyoune": "Autres",
    "Berrechid": "Autres", 
    "Deroua": "Autres",  
    "Had Soualem"  : "Autres",
    "Settat": "Autres",  
    "Benslimane": "Autres",
    "Ain Aouda" : "Autres", 
    "Tamesna-Ain Attig-Sidi Allal El Bahraoui":"Autres",
    "El Jadida": "Autres",
}

# Appliquer le mapping
df['Ville'] = df['Ville'].replace(mapping_villes)

# Vérifier le nouveau count
print(df['Ville'].value_counts())


In [ ]:
# Calculate the number of nulls in each column
nulls_per_column = df.isnull().sum()

# Calculate the total number of nulls in the DataFrame
total_nulls = nulls_per_column.sum()

# Calculate the total number of unique records that contain at least one null value
total_unique_records_with_nulls = len(df[df.isnull().any(axis=1)].drop_duplicates())

print("Nulls in each feature:")
print(nulls_per_column)
print("\nTotal nulls in the DataFrame:", total_nulls)
print("\nTotal records with at least one null value: ", total_unique_records_with_nulls)

In [ ]:
df.drop(['Surface_Habitable', 'Chambres', 'Salons', 'Condition'], axis=1, inplace=True)

In [ ]:
df.drop(columns=["Lien"], inplace=True, errors='ignore')
df = df.rename(columns={"Securite": "Lien"})

In [ ]:
df.columns

In [ ]:
df['Age'].unique()

In [ ]:
df['Age_encoded'] = df['Age'].map({'Moins de 1 an': 0, '1-5 ans': 1, '6-10 ans': 2, '11-20 ans': 3, '21+ ans': 4})
df.head()

In [ ]:
df['Standing'].unique()

In [ ]:
df['Standing_encoded'] = df['Standing'].map({'Neuf':0, 'Bon état':1, 'à rénover':2})
df.head()

In [ ]:
df['Etage'].unique()    

In [ ]:
df.drop(columns=['Etage'], inplace=True)

In [ ]:
df['Salles_De_Bain'].value_counts()

In [ ]:
# Convertit en numérique, remplace les erreurs (comme 'Nan') par NaN
df['Salles_De_Bain'] = pd.to_numeric(df['Salles_De_Bain'], errors='coerce')

df['Salles_De_Bain'] = df['Salles_De_Bain'].apply(lambda x: '2+' if x > 2 else x)
df['Salles_De_Bain'] = df['Salles_De_Bain'].replace({'2+': 3})


In [ ]:
df['Salles_De_Bain'].value_counts()

In [ ]:
# Function to convert a single price to Dirhams
def convert_to_dirhams(price):
    if isinstance(price, str):
        # Replace commas with periods and non-breaking spaces with nothing
        price = price.replace(',', '.').replace('\xa0', '')
        # Remove ' DH' and convert to float
        return float(price.replace('DH', '').replace(' ', ''))
    else:
        # If not a string, return as is
        return price

# Apply the function to each value in the 'Prix' column
df['Prix'] = df['Prix'].apply(convert_to_dirhams)

# Convert the non-null values in the 'Prix' column to integer
df.loc[df['Prix'].notna(), 'Prix'] = df.loc[df['Prix'].notna(), 'Prix'].astype(int)

df.head()

In [ ]:
columns = ['Parking', 'Chauffage', 'Climatisation']
for column in columns:
    df[column] = df[column].replace({"oui": 1, "Non": 0})
    
df.head()

In [ ]:

df['Age_encoded'] = df.groupby('Ville')['Age_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Age_encoded'] = df['Age_encoded'].fillna(df['Age_encoded'].median())
df['Age_encoded'] = df['Age_encoded'].round().astype(int)


In [ ]:
df['Standing_encoded'] = df.groupby('Ville')['Standing_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Standing_encoded'] = df['Standing_encoded'].fillna(df['Standing_encoded'].median())
df['Standing_encoded'] = df['Standing_encoded'].round().astype(int)

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
df['Salles_De_Bain'] = num_imputer.fit_transform(df[['Salles_De_Bain']])

In [ ]:
df.info()

In [ ]:
df['Surface_Totale'] = pd.to_numeric(df['Surface_Totale'], errors='coerce')
df['Surface_Totale'] = df['Surface_Totale'].astype(float)

In [ ]:
from rapidfuzz import fuzz, process

# Exemple : liste unique de quartiers
quartiers = df['Quartier'].unique()

# Dictionnaire de correspondance
corrections = {}

# Boucle pour comparer chaque quartier aux autres
for i, q1 in enumerate(quartiers):
    for q2 in quartiers[i+1:]:
        score = fuzz.ratio(q1.lower(), q2.lower())
        if score >= 85:  # seuil de similarité
            print(f"{q1} <--> {q2} : {score}% similaire")
            # On choisit la forme la plus courte comme forme standard
            standard = min(q1, q2, key=len)
            corrections[q1] = standard
            corrections[q2] = standard

print("\nDictionnaire de remplacement :")
print(corrections)


In [ ]:
# Fonction de remplacement
def corriger_quartier(q):
    return corrections.get(q, q)  # si pas dans le dict, garder original

df['Quartier'] = df['Quartier'].apply(corriger_quartier)


In [ ]:
# Calculer la moyenne du prix par quartier
quartier_mean = df.groupby('Quartier')['Prix'].mean()

# Associer chaque ligne à la moyenne de son quartier
df['Quartier_enc'] = df['Quartier'].map(quartier_mean)

In [ ]:
df['Prix'].describe()

<h2 style="color:black"> Using IQR </h2>

In [ ]:
df["Salles_De_Bain"] = pd.to_numeric(df["Salles_De_Bain"], errors="coerce")
Q1 = df["Salles_De_Bain"].quantile(0.25)
Q3 = df["Salles_De_Bain"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition0 = (df["Salles_De_Bain"] < (Q1 - 1.5 * IQR)) | (df["Salles_De_Bain"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliers = df[outlier_condition0]["Salles_De_Bain"]

# Print the number of outliers
print("\nNumber of outliers in Nombre De Salles_De_Bain:", outliers.count())

In [ ]:
Q1 = df["Prix"].quantile(0.25)
Q3 = df["Prix"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition = (df["Prix"] < (Q1 - 1.5 * IQR)) | (df["Prix"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQP = df[outlier_condition]

# Print the number of outliers
print("\nNumber of outliers in Prix:", outliersQP.shape[0])

In [ ]:
df = df[~outlier_condition]


In [ ]:
Q1 = df["Surface_Totale"].quantile(0.25)
Q3 = df["Surface_Totale"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition1 = (df["Surface_Totale"] < (Q1 - 1.5 * IQR)) | (df["Surface_Totale"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQS = df[outlier_condition1]
# Print the number of outliers
print("\nNumber of outliers in Surface:", outliersQS.shape[0])

In [ ]:
plt.figure(figsize=(4, 6))

# Subplot 1: Surface
plt.subplot(1, 2, 1)
sns.boxplot(y=df["Surface_Totale"], color='green')
plt.title('IQR of Surface')
plt.ylim(0, 600000)  # Set the y-axis limits from 0 to 1000

# Subplot 2:'Prix'
plt.subplot(1, 2, 2)
sns.boxplot(y=df["Prix"],width=0.6, color='orange')
plt.title('IQR of Prix')
plt.ylim(0, 9000000)  # Set the y-axis limits from 0 to 1000

plt.tight_layout()
plt.show()

In [ ]:
df['Prix'].describe()

In [ ]:
df = df[df['Prix'] >= 10000]
df.shape[0]

<h2 style="color:black"> Using z-score </h2>

In [ ]:
z_scores = np.abs(stats.zscore(df['Prix']))
threshold = 3
outliersZ = z_scores > threshold

# Print the outliers
print(df['Prix'][outliersZ].count())

In [ ]:
z_scores = np.abs(stats.zscore(df['Surface_Totale']))
threshold = 3
outliersZ = z_scores > threshold

# Print the outliers number
print(df['Prix'][outliersZ].count())

In [ ]:
outliersQP.describe()

In [ ]:
outliersQS.describe()

In [ ]:
# Exemple : le lien connu
lien_cible = "https://www.avito.ma/fr/ben_m_sick/local/Local_commercial_à_Casablanca__55321035.htm"

df.loc[df['Lien'] == lien_cible, 'Prix'] = 580000
df.loc[df['Lien'] == lien_cible, 'Surface_Totale'] = 30


In [ ]:
print(df[df['Lien'] == lien_cible][['Prix', 'Surface_Totale', 'Lien']])


In [ ]:
df = df.drop_duplicates(subset='Lien', keep='first')
df.shape[0]

In [ ]:
plt.figure(figsize=(6.6, 4.6))
plt.scatter( df['Surface_Totale'] ,df['Prix'] ,color='#0080ff', label='Training Data')
plt.xlabel("Surface_Totale")
plt.ylabel('Prix')
plt.xlim(0, 3000)
plt.ylim(0, 6000000)
plt.show()

In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO local_Transformed ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()
